# HW3 — Naive Bayes Experiments

Structured experimentation to push past the current best of **0.88676 CV** (GaussianNB + row-stats + QuantileTransformer). Each experiment is a controlled comparison; results land in a single summary table at the end. If any experiment beats the baseline, its predictions become the new canonical submission.

Experiments:
1. `n_quantiles` sweep for QuantileTransformer
2. Expanded row statistics (q25/q75/IQR, trimmed mean, L2 norm, count>0)
3. PT + QT prediction ensemble
4. Feature selection by univariate AUC threshold
5. BernoulliNB on quantile-discretized features
6. Mixture-of-Gaussians NB (hand-coded, per-class GMM per feature)

In [1]:
import os, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from sklearn.naive_bayes import GaussianNB, BernoulliNB
from sklearn.preprocessing import PowerTransformer, QuantileTransformer, KBinsDiscretizer
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.mixture import GaussianMixture

import warnings
warnings.filterwarnings('ignore')

DATA_DIR = 'data'
ART_DIR = 'artifacts'
RANDOM_STATE = 42
BASELINE_AUC = 0.88676  # GaussianNB + row-stats + QT (Section 10 of 02_naive_bayes.ipynb)
sns.set_style('whitegrid')

In [2]:
feature_cols = [f'var_{i}' for i in range(200)]
train_dtypes = {c: np.float32 for c in feature_cols} | {'target': np.int8}
test_dtypes = {c: np.float32 for c in feature_cols}

df_train = pd.read_csv(f'{DATA_DIR}/train.csv', dtype=train_dtypes)
df_test = pd.read_csv(f'{DATA_DIR}/test.csv', dtype=test_dtypes)
feature_diag = pd.read_csv(f'{ART_DIR}/feature_diagnostics.csv', index_col='feature')

X = df_train[feature_cols].values
y = df_train['target'].values
X_test = df_test[feature_cols].values
test_ids = df_test['ID_code'].values
print(f'X: {X.shape}, y: {y.shape}, X_test: {X_test.shape}')

X: (200000, 200), y: (200000,), X_test: (200000, 200)


In [3]:
def add_row_stats(X_arr, mode='basic'):
    """
    mode='basic' -> 8 aggregates (used by 02_naive_bayes.ipynb).
    mode='expanded' -> 14 aggregates (adds q25, q75, IQR, trimmed-mean, count>0, L2).
    """
    basic = [
        X_arr.mean(axis=1),
        X_arr.std(axis=1),
        X_arr.min(axis=1),
        X_arr.max(axis=1),
        np.median(X_arr, axis=1),
        X_arr.max(axis=1) - X_arr.min(axis=1),
        stats.skew(X_arr, axis=1),
        stats.kurtosis(X_arr, axis=1),
    ]
    if mode == 'expanded':
        q25 = np.percentile(X_arr, 25, axis=1)
        q75 = np.percentile(X_arr, 75, axis=1)
        basic += [
            q25, q75, q75 - q25,
            stats.trim_mean(X_arr, 0.1, axis=1),
            np.sum(X_arr > 0, axis=1).astype(X_arr.dtype),
            np.linalg.norm(X_arr, axis=1),
        ]
    extras = np.column_stack(basic)
    return np.hstack([X_arr, extras.astype(X_arr.dtype)])

X_aug = add_row_stats(X, 'basic')
X_test_aug = add_row_stats(X_test, 'basic')
print(f'Basic-augmented features: {X_aug.shape[1]}')

Basic-augmented features: 208


In [4]:
def cv_auc_general(X_arr, y_arr, model_factory, n_splits=5, seed=RANDOM_STATE, verbose=False):
    """Stratified k-fold CV with arbitrary model factory. Returns (aucs, oof)."""
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    aucs, oof = [], np.zeros(len(y_arr))
    for fold, (tr, va) in enumerate(skf.split(X_arr, y_arr), 1):
        m = model_factory()
        m.fit(X_arr[tr], y_arr[tr])
        p = m.predict_proba(X_arr[va])[:, 1]
        oof[va] = p
        aucs.append(roc_auc_score(y_arr[va], p))
        if verbose:
            print(f'  Fold {fold}: AUC = {aucs[-1]:.5f}')
    return np.array(aucs), oof

results = []
def log_result(name, aucs, notes=''):
    results.append({
        'experiment': name,
        'mean_auc': aucs.mean(),
        'std_auc': aucs.std(),
        'delta_vs_baseline': aucs.mean() - BASELINE_AUC,
        'notes': notes,
    })
    print(f'  -> {name}: mean AUC = {aucs.mean():.5f} (Δ vs baseline = '
          f'{aucs.mean() - BASELINE_AUC:+.5f})')

## Reference baseline (confirm)
Re-run the current best configuration to confirm we reproduce 0.88676.

In [5]:
qt = QuantileTransformer(output_distribution='normal', n_quantiles=1000,
                         subsample=200_000, random_state=RANDOM_STATE)
X_aug_qt = qt.fit_transform(X_aug).astype(np.float32)
X_test_aug_qt = qt.transform(X_test_aug).astype(np.float32)

baseline_aucs, baseline_oof = cv_auc_general(X_aug_qt, y, GaussianNB)
log_result('Baseline: GaussianNB + row-stats + QT(1000)', baseline_aucs)

  -> Baseline: GaussianNB + row-stats + QT(1000): mean AUC = 0.88676 (Δ vs baseline = -0.00000)


## Experiment 1 — `n_quantiles` sweep

In [6]:
exp1_rows = []
best_nq_aucs = None; best_nq = None
for nq in [200, 500, 1000, 2000, 5000]:
    qt_i = QuantileTransformer(output_distribution='normal', n_quantiles=nq,
                               subsample=200_000, random_state=RANDOM_STATE)
    Xt = qt_i.fit_transform(X_aug).astype(np.float32)
    aucs, _ = cv_auc_general(Xt, y, GaussianNB)
    exp1_rows.append((nq, aucs.mean(), aucs.std()))
    print(f'  n_quantiles={nq:>4d}  mean AUC={aucs.mean():.5f}  std={aucs.std():.5f}')
    if best_nq_aucs is None or aucs.mean() > best_nq_aucs.mean():
        best_nq_aucs, best_nq = aucs, nq
log_result(f'Exp1: QT n_quantiles={best_nq} (best in sweep)', best_nq_aucs)

  n_quantiles= 200  mean AUC=0.88741  std=0.00254
  n_quantiles= 500  mean AUC=0.88707  std=0.00259
  n_quantiles=1000  mean AUC=0.88676  std=0.00261
  n_quantiles=2000  mean AUC=0.88652  std=0.00263
  n_quantiles=5000  mean AUC=0.88632  std=0.00264
  -> Exp1: QT n_quantiles=200 (best in sweep): mean AUC = 0.88741 (Δ vs baseline = +0.00065)


## Experiment 2 — expanded row statistics (14 aggregates)

In [7]:
X_aug_v2 = add_row_stats(X, 'expanded')
X_test_aug_v2 = add_row_stats(X_test, 'expanded')
print(f'Expanded-augmented features: {X_aug_v2.shape[1]}')

qt_v2 = QuantileTransformer(output_distribution='normal', n_quantiles=1000,
                            subsample=200_000, random_state=RANDOM_STATE)
X_aug_v2_qt = qt_v2.fit_transform(X_aug_v2).astype(np.float32)
X_test_aug_v2_qt = qt_v2.transform(X_test_aug_v2).astype(np.float32)

exp2_aucs, exp2_oof = cv_auc_general(X_aug_v2_qt, y, GaussianNB)
log_result('Exp2: GaussianNB + expanded row-stats + QT', exp2_aucs)

Expanded-augmented features: 214
  -> Exp2: GaussianNB + expanded row-stats + QT: mean AUC = 0.88101 (Δ vs baseline = -0.00575)


## Experiment 3 — PT + QT prediction ensemble

In [8]:
pt = PowerTransformer(method='yeo-johnson', standardize=True)
X_aug_pt = pt.fit_transform(X_aug).astype(np.float32)
X_test_aug_pt = pt.transform(X_test_aug).astype(np.float32)

pt_aucs, pt_oof = cv_auc_general(X_aug_pt, y, GaussianNB)
print(f'  PT alone: {pt_aucs.mean():.5f}')
print(f'  QT alone: {baseline_aucs.mean():.5f}')

# 50/50 average
ens50 = (pt_oof + baseline_oof) / 2
ens50_auc = roc_auc_score(y, ens50)
print(f'  50/50 ensemble (whole-set OOF AUC): {ens50_auc:.5f}')

# Sweep weights
best_w, best_w_auc = 0.5, ens50_auc
for w in np.linspace(0, 1, 11):
    a = roc_auc_score(y, w * pt_oof + (1 - w) * baseline_oof)
    if a > best_w_auc:
        best_w, best_w_auc = w, a
print(f'  Best weight (PT={best_w:.1f}, QT={1-best_w:.1f}): {best_w_auc:.5f}')

# Express as per-fold AUC for consistency (mean across folds, ensemble within each fold)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
ens_aucs = []
for tr, va in skf.split(X_aug, y):
    p_pt = pt_oof[va]; p_qt = baseline_oof[va]
    ens_aucs.append(roc_auc_score(y[va], best_w * p_pt + (1 - best_w) * p_qt))
ens_aucs = np.array(ens_aucs)
log_result(f'Exp3: PT+QT ensemble (w_PT={best_w:.1f})', ens_aucs)

  PT alone: 0.88506
  QT alone: 0.88676
  50/50 ensemble (whole-set OOF AUC): 0.88640
  Best weight (PT=0.0, QT=1.0): 0.88677
  -> Exp3: PT+QT ensemble (w_PT=0.0): mean AUC = 0.88676 (Δ vs baseline = -0.00000)


## Experiment 4 — feature selection by univariate AUC

In [9]:
thresholds = [0.51, 0.515, 0.52, 0.525, 0.53]
best_thr_aucs, best_thr = None, None
for thr in thresholds:
    keep = feature_diag.index[feature_diag.univariate_auc > thr].tolist()
    if len(keep) < 10:
        continue
    X_keep = df_train[keep].values
    X_keep_aug = add_row_stats(X_keep, 'basic')
    qt_k = QuantileTransformer(output_distribution='normal', n_quantiles=1000,
                               subsample=200_000, random_state=RANDOM_STATE)
    Xt = qt_k.fit_transform(X_keep_aug).astype(np.float32)
    aucs, _ = cv_auc_general(Xt, y, GaussianNB)
    print(f'  threshold={thr}  kept={len(keep):>3d}/200  AUC={aucs.mean():.5f}')
    if best_thr_aucs is None or aucs.mean() > best_thr_aucs.mean():
        best_thr_aucs, best_thr = aucs, thr
log_result(f'Exp4: feature selection (univariate_auc > {best_thr})', best_thr_aucs)

  threshold=0.51  kept=155/200  AUC=0.88500
  threshold=0.515  kept=135/200  AUC=0.88279
  threshold=0.52  kept=117/200  AUC=0.87527
  threshold=0.525  kept= 92/200  AUC=0.86685
  threshold=0.53  kept= 70/200  AUC=0.85490
  -> Exp4: feature selection (univariate_auc > 0.51): mean AUC = 0.88500 (Δ vs baseline = -0.00176)


## Experiment 5 — BernoulliNB on quantile-discretized features

Bin each feature into `k` quantile bins, one-hot encode, then BernoulliNB. Tests a completely different inductive bias on the same data.

In [10]:
def cv_bnb(X_raw, y_arr, n_bins, n_splits=5):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    aucs, oof = [], np.zeros(len(y_arr))
    for tr, va in skf.split(X_raw, y_arr):
        disc = KBinsDiscretizer(n_bins=n_bins, encode='onehot', strategy='quantile')
        disc.fit(X_raw[tr])
        Xt = disc.transform(X_raw[tr])
        Xv = disc.transform(X_raw[va])
        m = BernoulliNB()
        m.fit(Xt, y_arr[tr])
        p = m.predict_proba(Xv)[:, 1]
        oof[va] = p
        aucs.append(roc_auc_score(y_arr[va], p))
    return np.array(aucs), oof

best_bnb_aucs, best_bnb_k = None, None
for k in [10, 20, 50]:
    t0 = time.time()
    aucs, _ = cv_bnb(X, y, n_bins=k)
    print(f'  k={k:>2d} bins  AUC={aucs.mean():.5f}  std={aucs.std():.5f}  '
          f'(time={time.time()-t0:.1f}s)')
    if best_bnb_aucs is None or aucs.mean() > best_bnb_aucs.mean():
        best_bnb_aucs, best_bnb_k = aucs, k
log_result(f'Exp5: BernoulliNB on {best_bnb_k}-bin quantile-discretized features',
           best_bnb_aucs)

  k=10 bins  AUC=0.87899  std=0.00285  (time=16.5s)
  k=20 bins  AUC=0.89046  std=0.00238  (time=18.8s)
  k=50 bins  AUC=0.88247  std=0.00262  (time=21.7s)
  -> Exp5: BernoulliNB on 20-bin quantile-discretized features: mean AUC = 0.89046 (Δ vs baseline = +0.00370)


## Experiment 6 — Mixture-of-Gaussians NB

Replace each per-class single-Gaussian density with a `k`-component Gaussian mixture. Still respects NB independence (one density per feature per class) but is much more flexible per-feature.

In [11]:
def fit_gmm_nb(X_tr, y_tr, n_components):
    n_feat = X_tr.shape[1]
    gmms = []
    for j in range(n_feat):
        per_class = []
        for cls in (0, 1):
            gm = GaussianMixture(n_components=n_components, max_iter=50,
                                 n_init=1, random_state=RANDOM_STATE,
                                 reg_covar=1e-6)
            gm.fit(X_tr[y_tr == cls, j:j+1])
            per_class.append(gm)
        gmms.append(per_class)
    n0 = (y_tr == 0).sum(); n1 = (y_tr == 1).sum()
    log_priors = np.array([np.log(n0 / (n0 + n1)), np.log(n1 / (n0 + n1))])
    return gmms, log_priors

def predict_gmm_nb(X_arr, gmms, log_priors):
    log_lik = np.broadcast_to(log_priors, (len(X_arr), 2)).copy()
    for j, (gm0, gm1) in enumerate(gmms):
        col = X_arr[:, j:j+1]
        log_lik[:, 0] += gm0.score_samples(col)
        log_lik[:, 1] += gm1.score_samples(col)
    return 1.0 / (1.0 + np.exp(log_lik[:, 0] - log_lik[:, 1]))

def cv_gmm_nb(X_arr, y_arr, n_components, n_splits=5):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    aucs, oof = [], np.zeros(len(y_arr))
    for tr, va in skf.split(X_arr, y_arr):
        gmms, lp = fit_gmm_nb(X_arr[tr], y_arr[tr], n_components)
        p = predict_gmm_nb(X_arr[va], gmms, lp)
        oof[va] = p
        aucs.append(roc_auc_score(y_arr[va], p))
    return np.array(aucs), oof

In [12]:
best_gmm_aucs, best_gmm_nc, best_gmm_oof = None, None, None
for nc in [1, 2, 3]:
    t0 = time.time()
    aucs, oof = cv_gmm_nb(X_aug_qt, y, n_components=nc)
    print(f'  n_components={nc}  AUC={aucs.mean():.5f}  std={aucs.std():.5f}  '
          f'(time={time.time()-t0:.1f}s)')
    if best_gmm_aucs is None or aucs.mean() > best_gmm_aucs.mean():
        best_gmm_aucs, best_gmm_nc, best_gmm_oof = aucs, nc, oof
log_result(f'Exp6: GMM-NB (n_components={best_gmm_nc})', best_gmm_aucs)

  n_components=1  AUC=0.88676  std=0.00261  (time=21.9s)
  n_components=2  AUC=0.89038  std=0.00257  (time=126.4s)
  n_components=3  AUC=0.89002  std=0.00221  (time=176.5s)
  -> Exp6: GMM-NB (n_components=2): mean AUC = 0.89038 (Δ vs baseline = +0.00362)


## Final results

In [13]:
results_df = pd.DataFrame(results).sort_values('mean_auc', ascending=False).reset_index(drop=True)
results_df

,experiment,mean_auc,std_auc,delta_vs_baseline,notes
0,Exp5: BernoulliNB on 20-bin quantile-discretiz...,0.890457,0.002383,3.697305e-03,
1,Exp6: GMM-NB (n_components=2),0.890378,0.002567,3.617516e-03,
2,Exp1: QT n_quantiles=200 (best in sweep),0.887409,0.002542,6.490011e-04,
3,Baseline: GaussianNB + row-stats + QT(1000),0.886760,0.002615,-2.586421e-07,
4,Exp3: PT+QT ensemble (w_PT=0.0),0.886760,0.002615,-2.586421e-07,
5,Exp4: feature selection (univariate_auc > 0.51),0.884999,0.002882,-1.760714e-03,
6,Exp2: GaussianNB + expanded row-stats + QT,0.881012,0.002472,-5.748388e-03,


In [14]:
winner = results_df.iloc[0]
print(f'Winner: {winner.experiment}')
print(f'Mean AUC: {winner.mean_auc:.5f}  (Δ vs baseline = {winner.delta_vs_baseline:+.5f})')

if winner.mean_auc > BASELINE_AUC + 1e-5:
    print('\nWinner beats baseline — writing submission_naive_bayes_v5.csv')
    # Reproduce the winning prediction on test set
    name = winner.experiment
    if name.startswith('Exp1'):
        qt_w = QuantileTransformer(output_distribution='normal', n_quantiles=best_nq,
                                   subsample=200_000, random_state=RANDOM_STATE)
        Xtr = qt_w.fit_transform(X_aug).astype(np.float32)
        Xte = qt_w.transform(X_test_aug).astype(np.float32)
        model = GaussianNB().fit(Xtr, y)
        proba = model.predict_proba(Xte)[:, 1]
    elif name.startswith('Exp2'):
        model = GaussianNB().fit(X_aug_v2_qt, y)
        proba = model.predict_proba(X_test_aug_v2_qt)[:, 1]
    elif name.startswith('Exp3'):
        m_pt = GaussianNB().fit(X_aug_pt, y); p_pt = m_pt.predict_proba(X_test_aug_pt)[:, 1]
        m_qt = GaussianNB().fit(X_aug_qt, y); p_qt = m_qt.predict_proba(X_test_aug_qt)[:, 1]
        proba = best_w * p_pt + (1 - best_w) * p_qt
    elif name.startswith('Exp4'):
        keep = feature_diag.index[feature_diag.univariate_auc > best_thr].tolist()
        Xs = df_train[keep].values; Xs_test = df_test[keep].values
        Xs_aug = add_row_stats(Xs, 'basic'); Xs_test_aug = add_row_stats(Xs_test, 'basic')
        qt_w = QuantileTransformer(output_distribution='normal', n_quantiles=1000,
                                   subsample=200_000, random_state=RANDOM_STATE)
        Xtr = qt_w.fit_transform(Xs_aug).astype(np.float32)
        Xte = qt_w.transform(Xs_test_aug).astype(np.float32)
        model = GaussianNB().fit(Xtr, y)
        proba = model.predict_proba(Xte)[:, 1]
    elif name.startswith('Exp5'):
        disc = KBinsDiscretizer(n_bins=best_bnb_k, encode='onehot', strategy='quantile')
        Xtr = disc.fit_transform(X)
        Xte = disc.transform(X_test)
        model = BernoulliNB().fit(Xtr, y)
        proba = model.predict_proba(Xte)[:, 1]
    elif name.startswith('Exp6'):
        gmms, lp = fit_gmm_nb(X_aug_qt, y, best_gmm_nc)
        proba = predict_gmm_nb(X_test_aug_qt, gmms, lp)
    else:
        raise RuntimeError(f'No reproducer wired up for {name}')
    pd.DataFrame({'ID_code': test_ids, 'target': proba}).to_csv(
        'submission_naive_bayes_v5.csv', index=False)
    print(f'Wrote submission_naive_bayes_v5.csv  mean={proba.mean():.4f}  std={proba.std():.4f}')
else:
    print('\nNo experiment beat the baseline. submission_naive_bayes_final.csv stays canonical.')

Winner: Exp5: BernoulliNB on 20-bin quantile-discretized features
Mean AUC: 0.89046  (Δ vs baseline = +0.00370)

Winner beats baseline — writing submission_naive_bayes_v5.csv
Wrote submission_naive_bayes_v5.csv  mean=0.0923  std=0.1677
